# One frozen candidate, fresh grouped D confirmation — v011

Definitions and review only by default. No approved data is read, no D split is
constructed, and no model is fitted unless `PA3_RUN_CONFIRM_V011=1` is explicit.
Production execution additionally requires the exact notebook **code** hash in
`PA3_CONFIRM_V011_SOURCE_SHA256`; checkpoint recovery uses
`PA3_RESUME_CONFIRM_V011=1`. Use the normal approved course GPU allocation.

After **both** v009 arms and the two predeclared v010 mixtures and the single predeclared v012 mixture have completed,
select the single lowest-A candidate meeting the unchanged .003 equal-pair gain
and wholly negative whole-breath bootstrap interval. No extra weights or rosters
are searched. Before parsing any D pressure label or reading a D outcome, freeze
that choice, weights, every member's A-selected rate prefix, source/data/artifact
hashes and the new RC-stratified groups in `locked_before_D.json`. This immutable
lock has no changing creation timestamp. Existing locks must match exactly.

D is drawn inside original Afit only: sklearn `train_test_split`, test_size6791,
random_state20261202, joint numeric R/C strata; sorted global indices give
54,323 fitting and 6,791 validation breaths. The original A validation is excluded
from every D fit and D score. D is **resampling confirmation**, not a pristine
untouched test. The split reads only known input IDs and R/C before the lock.

Cold-fit the compact193 and deep189 reference members plus any newly selected
members on Dfit, each with its own training-only statistics and frozen A-selected
rates. Original A checkpoints are provenance only and are never loaded for D
prediction. The unchanged V005/V008 engines retain Adam/shuffling/RNG/exact-resume
protections. The fixed-budget fit functions receive no validation arrays.

All required cold fits finish before native D metrics are evaluated. Confirmation
passes only if candidate gain is strictly positive and the point-weighted,
whole-breath 1,000-draw bootstrap upper95% bound is below zero. D cannot choose
weights, epochs, a runner-up or a replacement candidate. A failed confirmation
ends this recipe. This notebook contains **no final-all fitting or submission**;
accepted .18550 artifacts remain preserved. A separate notebook may consume the
immutable lock and successful confirmation result.


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import copy
import fcntl
import gc
import hashlib
import json
import math
import tempfile
import uuid
from pathlib import Path
import numpy as np
import pandas as pd

V011_RUN = os.environ.get('PA3_RUN_CONFIRM_V011', '0') == '1'
V011_RESUME = os.environ.get('PA3_RESUME_CONFIRM_V011', '0') == '1'
V011_ROOT = Path.cwd().resolve()
V011_NOTEBOOK = 'confirmation_v011.ipynb'
V011_OUT = 'outputs/confirmation_v011'
V011_COMPONENTS = {'feature_components_v005.ipynb': 'eb19eed69eb54cd5d087a9a678cf98701fbe9bc1a3cf85016270754c014593a6', 'sequence_components_v005.ipynb': 'e6698a7f6e931af8a066c227058203b7e29ada68624833b700293532a30bf1b2', 'training_components_v008.ipynb': 'a65618d1886a7ad3d1753f1f752b449114a57dcb75084f4f6cd11fc084228bda', 'attention_components_v008.ipynb': '47c43f75bf58385771de89e37b0fed05054f8a6d6e5bd1ac3e15f4891111625b', 'memory_components_v008.ipynb': 'b7dd6c6434ada27ae5cee3e098aa113614ac891e0ecd4e7e25917ab92ad47d8a', 'offset_components_v009.ipynb': '1000c5311fcf4cfc348fd5b99975fcd5816650d1616ace31eace5407ccd2a1b3', 'tcn_components_v009.ipynb': 'd4e08e56a78a8d1d8e45ee0843302d7d2a1c9b35ccc1ed8628ae4f1239da486e', 'tcn_fast_components_v009.ipynb': '51d7438d2bd719e5609ed8530a4ad9cf9009943498641bf9488c9e81b1a5024d'}
V011_COMPONENT_ORDER = tuple(V011_COMPONENTS)
V011_PORTABLE_A_NOTEBOOKS = frozenset('research/'+name for name in V011_COMPONENTS) | {
    'controlled_experiments_v008.ipynb', 'controlled_experiments_v009.ipynb',
    'iterative_improvements_v005.ipynb', 'research/diversity_screen_v010.ipynb',
    'research/diversity_screen_v012.ipynb'}
V011_MEMBER_IDS = ('original_compact', 'deep', 'curve', 'memory', 'attention', 'offset', 'tcn')
V011_A_ROSTER = {
    'deep_half_compact_curve_memory': (('deep', .5), ('original_compact', 1/6), ('curve', 1/6), ('memory', 1/6)),
    'deep_half_compact_curve_memory_attention': (('deep', .5), ('original_compact', 1/8), ('curve', 1/8), ('memory', 1/8), ('attention', 1/8)),
    'shared_breath_offset': (('deep', .5), ('offset', .5)),
    'full_context_tcn': (('deep', .5), ('tcn', .5)),
    'deep_half_compact_curve_memory_offset': (('deep', .5), ('original_compact', .125), ('curve', .125), ('memory', .125), ('offset', .125)),
}
V011_RAW_DTYPES = dict(id='int64', breath_id='int32', R='int8', C='int8',
    time_step='float32', u_in='float32', u_out='int8', pressure='float32')


def v011_read(path):
    return json.loads(Path(path).read_text())


def v011_file_sha(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            result.update(block)
    return result.hexdigest()


def v011_array_sha(values):
    return hashlib.sha256(np.ascontiguousarray(values).tobytes()).hexdigest()


def v011_code_sha(path):
    book = v011_read(path)
    return hashlib.sha256('\n'.join(''.join(cell.get('source', []))
        for cell in book['cells'] if cell.get('cell_type') == 'code').encode()).hexdigest()


def v011_digest(value):
    """External compact canonical JSON digest; never put this inside its own object."""
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()).hexdigest()


def v011_json_bytes(value):
    return (json.dumps(value, sort_keys=True, indent=2, allow_nan=False)+'\n').encode()


def v011_immutable_json(path, value):
    """Atomic publish without overwrite; existing immutable content must match."""
    path = Path(path)
    encoded = v011_json_bytes(value)
    if path.exists():
        if path.read_bytes() != encoded:
            raise ValueError('Immutable artifact differs; refusing overwrite: '+str(path))
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
    try:
        with temporary.open('xb') as stream:
            stream.write(encoded); stream.flush(); os.fsync(stream.fileno())
        try:
            os.link(temporary, path)
        except FileExistsError:
            if path.read_bytes() != encoded:
                raise ValueError('Concurrent immutable artifact differs: '+str(path))
    finally:
        temporary.unlink(missing_ok=True)


def v011_atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
    try:
        temporary.write_bytes(v011_json_bytes(value)); os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def v011_save_array(path, values):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
    try:
        with temporary.open('xb') as stream:
            np.save(stream, values, allow_pickle=False)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def v011_local(root, saved_path):
    value = str(saved_path); marker = '/'+root.name+'/'
    path = (root/value.split(marker, 1)[1] if marker in value else root/value).resolve()
    if not path.is_relative_to(root) or not path.is_file():
        raise ValueError('Missing artifact or path outside repository: '+value)
    return path


def v011_relative(root, path):
    return str(Path(path).resolve().relative_to(root))


def v011_require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def v011_source_guard(root, manifest=V011_COMPONENTS):
    for name, expected in manifest.items():
        v011_require(v011_code_sha(root/'research'/name) == expected, 'Component source changed: '+name)


def v011_load_components(root):
    v011_source_guard(root)
    namespace = {'__name__': '__main__'}
    for name in V011_COMPONENT_ORDER:
        book = v011_read(root/'research'/name)
        for index, cell in enumerate(book['cells']):
            if cell.get('cell_type') == 'code':
                exec(compile(''.join(cell.get('source', [])), name+':cell'+str(index), 'exec'), namespace)
    return namespace


def v011_factory(member_id, namespace):
    if member_id in ('original_compact', 'deep', 'curve'):
        constructor = namespace['V5PressureRNN']
        return lambda spec, n_features, prep: constructor(spec, n_features)
    if member_id == 'attention':
        constructor = namespace['v8_attention_model']
        return lambda spec, n_features, prep: constructor(spec, n_features)
    constructors = dict(memory='v8_memory_model', offset='v9_offset_model', tcn='v9_fast_tcn_model')
    if member_id not in constructors:
        raise ValueError('Unknown frozen factory ID: '+member_id)
    constructor = namespace[constructors[member_id]]
    return lambda spec, n_features, prep: constructor(spec, n_features, prep)


The next helpers inspect completed A artifacts only. The five candidate identities
and their weights are explicit constants. Member recipes retain the exact original
training configuration and the A-selected prefix; the fixed fitter, rather than
D outcomes, determines how many epochs run. Every referenced source/record/model/
prediction/report is verified and enters the immutable selection provenance.
Declared A notebook sources may differ only in outputs or metadata when their
executable code hash matches exactly. Expected/observed raw hashes are recorded;
the lock then strictly freezes the observed project files. All non-notebook data,
model, report and array files retain exact full-file hashes.


In [ ]:
def v011_member_recipe(root, member_id, record_path, inputs):
    record_path = Path(record_path); record = v011_read(record_path)
    checkpoint = v011_local(root, record['checkpoint_path'])
    prediction = v011_local(root, record['prediction_file'])
    report_path = v011_local(root, record['report_file'])
    report = v011_read(report_path)
    v011_require(v011_file_sha(checkpoint) == record['checkpoint_sha256'], 'A checkpoint hash: '+member_id)
    if 'report_sha256' in record:
        v011_require(v011_file_sha(report_path) == record['report_sha256'], 'A report hash: '+member_id)
    values = np.load(prediction, allow_pickle=False)
    v011_require(values.shape == (206098,) and values.dtype == np.float64 and np.isfinite(values).all()
        and v011_array_sha(values) == record['prediction_sha256'], 'A native predictions: '+member_id)
    spec, cfg = record['recipe']['model_spec'], record['recipe']['train_cfg']
    epoch = int(record['best_epoch']); rates = list(record['selected_lr_schedule'])
    v011_require(report['completed'] and report['executed_epochs'] == 200
        and record.get('executed_epochs', 200) == 200, 'Incomplete A stage: '+member_id)
    v011_require(report['checkpoint_sha256'] == record['checkpoint_sha256']
        and report['best_epoch'] == epoch and report['validation_mae'] == record['validation_mae'],
        'A report/model/metric mismatch: '+member_id)
    v011_require(1 <= epoch <= 200 and rates == cfg['frozen_lr_schedule'][:epoch]
        and len(rates) == epoch and report['executed_lr_schedule'] == cfg['frozen_lr_schedule']
        and report['selected_lr_schedule'] == rates, 'A selected rate prefix changed: '+member_id)
    v011_require(cfg['max_epochs'] == 200 and cfg['batch_size'] == 256 and cfg['optimizer'] == 'Adam'
        and cfg['weight_decay'] == 0. and cfg['loss'] == 'l1' and cfg['scaler_phase'] == 'all'
        and not cfg['residual_baseline'] and record['seed'] == 20261102
        and record['training_breaths'] == 61114, 'Original controlled recipe changed: '+member_id)
    if member_id == 'original_compact':
        v011_require(epoch == 193, 'Original compact prefix must be193')
    if member_id == 'deep':
        v011_require(epoch == 189, 'Original deep prefix must be189')
    for path in (record_path, checkpoint, prediction, report_path):
        inputs[v011_relative(root, path)] = v011_file_sha(path)
    return dict(member_id=member_id, factory_id=member_id,
        kind='v5' if member_id in ('original_compact', 'deep') else 'v8',
        recipe=dict(model_spec=copy.deepcopy(spec), train_cfg=copy.deepcopy(cfg), seed=20261102,
            feature_mode='combined', loss_kind='curve_l1' if member_id == 'curve' else 'l1',
            selected_epochs=epoch, lr_schedule=rates), source_fingerprints=copy.deepcopy(V011_COMPONENTS),
        A_artifacts=dict(record_path=v011_relative(root, record_path), record_sha256=v011_file_sha(record_path),
            checkpoint_path=v011_relative(root, checkpoint), checkpoint_sha256=record['checkpoint_sha256'],
            report_path=v011_relative(root, report_path), report_sha256=v011_file_sha(report_path),
            prediction_file=v011_relative(root, prediction), prediction_file_sha256=v011_file_sha(prediction),
            prediction_array_sha256=record['prediction_sha256']))


def v011_verify_screen_input(root, relative_path, expected_raw_sha256, source_manifest):
    """Allow metadata-only drift solely for whitelisted declared A notebook sources.

    Every other artifact keeps an exact full-file hash. Return the actual bytes'
    identity so the eventual pre-D lock freezes this project's observed files.
    """
    root = Path(root).resolve()
    relative_path = str(relative_path)
    path = v011_local(root, relative_path)
    observed_raw_sha256 = v011_file_sha(path)
    declared = source_manifest.get(relative_path)
    portable = relative_path in V011_PORTABLE_A_NOTEBOOKS and declared is not None
    if portable:
        v011_require(relative_path.endswith('.ipynb')
            and declared['file_sha256'] == expected_raw_sha256,
            'Declared notebook raw provenance differs: '+relative_path)
        observed_code_sha256 = v011_code_sha(path)
        v011_require(observed_code_sha256 == declared['code_sha256'],
            'Notebook executable code changed: '+relative_path)
        return dict(validation='exact_executable_code', expected_raw_file_sha256=expected_raw_sha256,
            observed_raw_file_sha256=observed_raw_sha256,
            expected_code_sha256=declared['code_sha256'], observed_code_sha256=observed_code_sha256,
            metadata_only_raw_difference=observed_raw_sha256 != expected_raw_sha256)
    v011_require(observed_raw_sha256 == expected_raw_sha256,
        'Strict nonportable artifact hash changed: '+relative_path)
    return dict(validation='strict_full_file_sha256', expected_raw_file_sha256=expected_raw_sha256,
        observed_raw_file_sha256=observed_raw_sha256)


def v011_pick_lowest_A_candidate(candidates):
    """The complete five-case set is frozen before any D input or outcome."""
    v011_require(len(candidates) == 5 and {row['name'] for row in candidates} == set(V011_A_ROSTER),
        'Exactly the five predeclared A candidates must finish')
    for row in candidates:
        eligible = row['A_gain'] >= .003 and row['A_interval']['bootstrap_95_high'] < 0.
        v011_require(abs(.19656260982217663-row['A_MAE']-row['A_gain']) < 1e-12
            and eligible == row['declared_eligible'], 'Frozen A qualification mismatch')
        row['eligible'] = bool(eligible)
    qualifiers = [row for row in candidates if row['eligible']]
    return min(qualifiers, key=lambda row: (row['A_MAE'], row['name'])) if qualifiers else None


def v011_completed_A_selection(root):
    """Finish all five fixed A comparisons before choosing; reads no D data."""
    root = Path(root).resolve(); v011_source_guard(root)
    base = root/'outputs/improvements_v005/sequence'
    a8 = root/'outputs/controlled_experiments_v008'
    a9 = root/'outputs/controlled_experiments_v009'
    summary8, summary9 = v011_read(a8/'screen_summary.json'), v011_read(a9/'screen_summary.json')
    diversity = v011_read(root/'outputs/diversity_screen_v010/results.json')
    diversity12 = v011_read(root/'outputs/diversity_screen_v012/results.json')
    review12 = v011_read(root/'research/diversity_review_v012.json')
    audit = v011_read(root/'research/screen_review_v008.json')
    review10 = v011_read(root/'research/diversity_review_v010.json')
    design8, design9, original = v011_read(a8/'design.json'), v011_read(a9/'design.json'), v011_read(base/'design.json')
    inputs = {}
    for path in (a8/'screen_summary.json', a9/'screen_summary.json', a8/'design.json', a9/'design.json',
            base/'design.json', root/'outputs/diversity_screen_v010/results.json',
            root/'research/screen_review_v008.json', root/'research/diversity_review_v010.json',
            root/'research/ENSEMBLE_HYPOTHESES_v010.md', root/'research/authorization_v008.json',
            root/'research/authorization_v009.json', base/'split_breath_ids.npz',
            root/'research/ENSEMBLE_HYPOTHESIS_v012.md', root/'research/diversity_screen_v012.ipynb',
            root/'outputs/diversity_screen_v012/results.json', root/'research/diversity_review_v012.json',
            root/'research/diversity_screen_v010.ipynb'):
        inputs[v011_relative(root, path)] = v011_file_sha(path)
    v011_require(summary8['status'] == summary9['status'] == 'complete'
        and summary8['selected_for_confirmation'] is None, 'Prior complete screens required')
    v011_require(audit['passed'] and audit['check_count'] == 403 and not audit['failed_checks'], 'V008 independent audit required')
    v011_require(review10['status'] == 'passed' and review10['check_count'] == 58
        and review10['results_sha256'] == v011_file_sha(root/'outputs/diversity_screen_v010/results.json'),
        'V010 independent fixed-screen verification required')
    v011_require(diversity['status'] == 'complete_provisional_no_promotion'
        and diversity['candidate_count'] == len(diversity['results']) == 2
        and diversity['selected_for_confirmation'] is None and not diversity['D_split_accessed_or_constructed'],
        'Two complete unconfirmed diversity candidates required')
    v011_require(diversity['code_hash_before_run'] == diversity['code_hash_after_run']
        == v011_code_sha(root/'research/diversity_screen_v010.ipynb'), 'V010 executable source changed')
    v011_require(diversity['hypotheses_sha256'] == v011_file_sha(root/'research/ENSEMBLE_HYPOTHESES_v010.md'),
        'Predeclared mixture hypotheses changed')
    notebook_portability_evidence = {}
    for path, item in diversity['verified_inputs'].items():
        evidence = v011_verify_screen_input(root, path, item['file_sha256'], diversity['source_manifest'])
        # Freeze every ACTUAL observed raw file digest after the initial validation.
        # Subsequent lock matching remains strict even for notebook metadata.
        inputs[path] = evidence['observed_raw_file_sha256']
        if evidence['validation'] == 'exact_executable_code':
            notebook_portability_evidence[path] = evidence
    v011_require(diversity12['schema'] == 'fixed_diversity_screen_v012'
        and diversity12['status'] == 'complete_provisional_no_promotion'
        and diversity12['candidate_count'] == len(diversity12['results']) == 1
        and not diversity12['weight_or_roster_search'] and diversity12['checks_passed'] == len(diversity12['checks']) == 181
        and diversity12['selected_for_confirmation'] is None and not diversity12['D_split_accessed_or_constructed'],
        'The one complete predeclared V012 mixture is required')
    v011_require(review12['status'] == 'passed' and review12['check_count'] > 0
        and all(value is True for value in review12['checks'].values())
        and review12['results_sha256'] == v011_file_sha(root/'outputs/diversity_screen_v012/results.json')
        and review12['candidate_name'] == 'deep_half_compact_curve_memory_offset', 'V012 independent verification required')
    v011_require(diversity12['hypotheses_file'] == 'research/ENSEMBLE_HYPOTHESIS_v012.md'
        and diversity12['hypotheses_sha256'] == review12['hypothesis_sha256']
        == v011_file_sha(root/'research/ENSEMBLE_HYPOTHESIS_v012.md'), 'V012 predeclaration changed')
    v011_require(diversity12['prior_v010_independent_review']['file_sha256']
        == v011_file_sha(root/'research/diversity_review_v010.json')
        and diversity12['completed_v009_manifest_sha256']
        == v011_file_sha(root/'outputs/controlled_experiments_v009/completed-backup-manifest.json'),
        'V012 prior screen/archive provenance changed')
    v011_require(diversity12['code_hash_before_run'] == diversity12['code_hash_after_run']
        == review12['source_code_sha256'] == v011_code_sha(root/'research/diversity_screen_v012.ipynb'),
        'V012 executable source changed')
    v011_require(diversity12['gate'] == dict(minimum_gain=.003,
        whole_breath_bootstrap_upper95_below_zero=True, seed=20261201, repeats=1000), 'V012 promotion rule changed')
    # Merge only explicit declarations. Duplicate paths must agree on executable
    # code; select the raw provenance declaration matching this report's input.
    manifests = (diversity['source_manifest'], diversity12['source_manifest'])
    for path in set(manifests[0]) & set(manifests[1]):
        v011_require(manifests[0][path]['code_sha256'] == manifests[1][path]['code_sha256'],
            'Declared source implementations disagree: '+path)
    for path, item in diversity12['verified_inputs'].items():
        declarations = [manifest[path] for manifest in manifests if path in manifest
            and manifest[path]['file_sha256'] == item['file_sha256']]
        declared = {path: declarations[0]} if declarations else {}
        v011_require(not any(path in manifest for manifest in manifests) or bool(declarations),
            'V012 input raw provenance lacks an exact declaration: '+path)
        evidence = v011_verify_screen_input(root, path, item['file_sha256'], declared)
        inputs[path] = evidence['observed_raw_file_sha256']
        if evidence['validation'] == 'exact_executable_code':
            notebook_portability_evidence[path] = evidence
    for notebook_path, code_hash in [('research/diversity_screen_v010.ipynb', diversity['code_hash_after_run']),
            ('research/diversity_screen_v012.ipynb', diversity12['code_hash_after_run'])]:
        observed_code = v011_code_sha(root/notebook_path)
        v011_require(observed_code == code_hash, 'Screen source changed: '+notebook_path)
        inputs[notebook_path] = v011_file_sha(root/notebook_path)
        notebook_portability_evidence[notebook_path] = dict(validation='exact_reported_executable_code',
            expected_code_sha256=code_hash, observed_code_sha256=observed_code,
            observed_raw_file_sha256=inputs[notebook_path])
    for filename, expected in design8['component_hashes'].items():
        v011_require(V011_COMPONENTS[filename] == expected, 'V008 source manifest differs')
    for filename, expected in design9['component_hashes'].items():
        v011_require(V011_COMPONENTS[filename] == expected, 'V009 source manifest differs')
    v011_require(v011_code_sha(root/'controlled_experiments_v008.ipynb') == design8['source_sha256']
        and v011_code_sha(root/'controlled_experiments_v009.ipynb') == design9['source_sha256'],
        'A orchestration source changed')
    v011_require(design8['train_sha256'] == design9['train_sha256'] == original['train_sha256'], 'Approved data identity mismatch')
    approval = v011_read(root/'research/authorization_v008.json')
    v011_require(approval['standing_approval_for_controlled_iterations'] and approval['only_assignment_approved_data'], 'Standing approval missing')
    expected_9 = {'shared_breath_offset', 'full_context_tcn'}
    v011_require(set(row['name'] for row in summary9['results']) == expected_9
        and len(summary9['results']) == 2 and design9['plan']['epochs'] == 200
        and design9['plan']['tcn_backend'] == 'shifted_linear', 'Both frozen V009 stages must finish')
    reference = .19656260982217663
    v011_require(abs(summary9['reference_A_pair_MAE']-reference) < 1e-12
        and abs(diversity['reference']['metrics']['mae']-reference) < 1e-12, 'Immutable A reference differs')
    aliases = dict(original_deep='deep', original_compact='original_compact',
        curve_consistency='curve', learned_memory='memory', residual_attention='attention', shared_breath_offset='offset')
    candidates = []
    for row in summary9['results']:
        path = a9/('record_A_'+row['name']+'.json')
        v011_require(v011_file_sha(path) == row['record_sha256'], 'V009 summary record hash differs')
        record = v011_read(path)
        v011_require(abs(record['pair_validation_mae']-row['fixed_pair_A_MAE']) < 1e-12
            and record['weights'] == [.5, .5] and record['partner'] == 'deep', 'V009 fixed pair differs')
        candidates.append(dict(name=row['name'], source_screen='v009', A_MAE=float(row['fixed_pair_A_MAE']),
            A_gain=float(row['gain']), A_interval=copy.deepcopy(row['interval']),
            declared_eligible=bool(row['eligible_for_fresh_grouped_confirmation'])))
    expected_10 = {'deep_half_compact_curve_memory', 'deep_half_compact_curve_memory_attention'}
    v011_require(set(row['name'] for row in diversity['results']) == expected_10, 'Unexpected mixture roster')
    for row in diversity['results']:
        weights = {aliases[name]: weight for name, weight in row['weights'].items()}
        v011_require(weights == dict(V011_A_ROSTER[row['name']]), 'Unfrozen mixture weights')
        candidates.append(dict(name=row['name'], source_screen='v010', A_MAE=float(row['metrics']['mae']),
            A_gain=float(row['gain_vs_original_pair']), A_interval=copy.deepcopy(row['interval']),
            declared_eligible=bool(row['qualifies_provisionally'])))
    row12 = diversity12['results'][0]
    v011_require(row12['name'] == 'deep_half_compact_curve_memory_offset'
        and {aliases[name]: weight for name, weight in row12['weights'].items()} == dict(V011_A_ROSTER[row12['name']])
        and row12['metrics'] == review12['metrics'] and row12['gain_vs_original_pair'] == review12['gain']
        and all(row12['interval'][key] == value for key, value in review12['interval'].items()),
        'V012 fixed recipe or independently verified evidence changed')
    for path_key, hash_key in [('prediction_file', 'prediction_file_sha256'), ('row_ids_file', 'row_ids_file_sha256')]:
        path = v011_local(root, row12[path_key])
        v011_require(v011_file_sha(path) == row12[hash_key], 'V012 saved prediction/row file changed')
        inputs[row12[path_key]] = v011_file_sha(path)
    v011_require(v011_array_sha(np.load(v011_local(root, row12['prediction_file']), allow_pickle=False))
        == row12['prediction_raw_array_sha256']
        and v011_array_sha(np.load(v011_local(root, row12['row_ids_file']), allow_pickle=False))
        == row12['row_ids_raw_array_sha256'], 'V012 native prediction/row array changed')
    candidates.append(dict(name=row12['name'], source_screen='v012', A_MAE=float(row12['metrics']['mae']),
        A_gain=float(row12['gain_vs_original_pair']), A_interval=copy.deepcopy(row12['interval']),
        declared_eligible=bool(row12['qualifies_provisionally'])))
    winner = v011_pick_lowest_A_candidate(candidates)
    if winner is None:
        return dict(winner=None, candidates=candidates, selection_input_files=inputs,
            notebook_portability_evidence=notebook_portability_evidence)
    records = dict(original_compact=base/'record_A_capacity_bilstm128.json',
        deep=base/'record_A_capacity_deep_bilstm.json', curve=a8/'record_A_curve_consistency.json',
        memory=a8/'record_A_learned_memory.json', attention=a8/'record_A_residual_attention.json',
        offset=a9/'record_A_shared_breath_offset.json', tcn=a9/'record_A_full_context_tcn.json')
    # Verify completion/provenance of BOTH new A members even if neither wins.
    all_recipes = {member_id: v011_member_recipe(root, member_id, path, inputs)
                   for member_id, path in records.items()}
    compact = all_recipes['original_compact']['recipe']
    for member_id, member in all_recipes.items():
        if member_id != 'deep':
            v011_require(member['recipe']['model_spec'] == compact['model_spec']
                and member['recipe']['train_cfg'] == compact['train_cfg'], 'Controlled compact recipe differs')
    # These are saved actual-device proofs, not newly launched GPU tests.
    for stage in (a8, a9):
        proof_path, profile_path = stage/'course_proofs.json', stage/'course_capacity_profiles.json'
        proofs, profiles = v011_read(proof_path), v011_read(profile_path)
        for proof_name, proof in proofs.items():
            proof_device = str(proof.get('device', proof.get('selected_device', '')))
            v011_require(proof['passed'] and proof_device.startswith('cuda')
                and all(value is True for value in proof['checks'].values()), 'Saved actual-CUDA proof failed')
            if proof_name.endswith('_resume'):
                v011_require(proof['gpu_checked'] and proof['runtime']['device_type'] == 'cuda', 'Saved CUDA exact-resume proof missing')
        for name, profile in profiles.items():
            v011_require(profile['strict_determinism'] and profile['batch_size'] == 256
                and profile['inference_batch_size'] == 512, 'Saved fixed-batch GPU profile differs')
            if name != 'baseline':
                v011_require(profile['passed'] and profile['relative_step_seconds'] <= 2.
                    and profile['peak_allocated_bytes'] < 9*1024**3, 'Saved GPU capacity gate failed')
        inputs[v011_relative(root, proof_path)] = v011_file_sha(proof_path)
        inputs[v011_relative(root, profile_path)] = v011_file_sha(profile_path)
    winner = copy.deepcopy(winner)
    winner['members'] = [dict(copy.deepcopy(all_recipes[name]), weight=weight)
                         for name, weight in V011_A_ROSTER[winner['name']]]
    baseline = dict(members=[dict(copy.deepcopy(all_recipes['original_compact']), weight=.5),
                             dict(copy.deepcopy(all_recipes['deep']), weight=.5)])
    required_ids = set(name for name, _ in V011_A_ROSTER[winner['name']]) | {'original_compact', 'deep'}
    required = [copy.deepcopy(all_recipes[name]) for name in V011_MEMBER_IDS if name in required_ids]
    return dict(winner=winner, baseline=baseline, required_members=required, candidates=candidates,
        selection_input_files=inputs, notebook_portability_evidence=notebook_portability_evidence,
        approved_train_sha256=original['train_sha256'],
        original_group_hashes=copy.deepcopy(original['group_hashes']))


Only IDs/R/C enter the split and lock phase. The public split helper imports
sklearn lazily, so default review needs no sklearn or Torch import. Group hashes
are raw bytes of int32 breath IDs; index hashes are raw bytes of sorted int64 global
indices. Array hashes in prediction artifacts are raw contiguous bytes, distinct
from engine fingerprints that also encode shape/dtype.


In [ ]:
def v011_split_from_known_metadata(breath_ids, rc, original_fit, original_validation):
    from sklearn.model_selection import train_test_split
    breath_ids = np.asarray(breath_ids, dtype=np.int32)
    rc = np.asarray(rc)
    original_fit = np.sort(np.asarray(original_fit, dtype=np.int64))
    original_validation = np.sort(np.asarray(original_validation, dtype=np.int64))
    v011_require(len(original_fit) == 61114 and len(original_validation) == 6791
        and not np.intersect1d(original_fit, original_validation).size, 'Original A groups invalid')
    codes = 3*np.searchsorted(np.array([5, 20, 50]), rc[original_fit, 0]) + np.searchsorted(np.array([10, 20, 50]), rc[original_fit, 1])
    fit, validation = train_test_split(original_fit, test_size=6791, random_state=20261202, stratify=codes)
    groups = dict(A_fit=original_fit, A_validation=original_validation,
                  D_fit=np.sort(fit.astype(np.int64)), D_validation=np.sort(validation.astype(np.int64)))
    v011_require(len(groups['D_fit']) == 54323 and len(groups['D_validation']) == 6791
        and not np.intersect1d(groups['D_fit'], groups['D_validation']).size
        and np.array_equal(np.sort(np.r_[groups['D_fit'], groups['D_validation']]), original_fit)
        and not np.intersect1d(np.r_[groups['D_fit'], groups['D_validation']], original_validation).size,
        'D groups violate frozen counts or original A exclusion')
    return dict(seed=20261202, method='sklearn.train_test_split(test_size=6791,random_state=20261202,stratify=joint_RC)',
        indices={key: values.tolist() for key, values in groups.items()},
        breath_ids={key: breath_ids[values].tolist() for key, values in groups.items()},
        group_hashes={key: v011_array_sha(breath_ids[values].astype(np.int32)) for key, values in groups.items()},
        index_hashes={key: v011_array_sha(values.astype(np.int64)) for key, values in groups.items()},
        counts={key: len(values) for key, values in groups.items()})


def v011_freeze_before_D(root, selection):
    """This phase never parses pressure, u_out or existing D outcomes."""
    root = Path(root).resolve(); out = root/V011_OUT
    v011_require(selection['winner'] is not None, 'No qualifying A candidate; no D work is authorized')
    v011_require(v011_file_sha(root/'data/train.csv') == selection['approved_train_sha256'], 'Approved train hash differs')
    metadata = pd.read_csv(root/'data/train.csv', usecols=['id', 'breath_id', 'R', 'C'],
        dtype={key: V011_RAW_DTYPES[key] for key in ('id', 'breath_id', 'R', 'C')})
    v011_require(len(metadata) == 67905*80 and metadata.id.is_unique, 'Known-input metadata row count/IDs invalid')
    blocks = metadata.breath_id.to_numpy().reshape(-1, 80)
    breath_ids = blocks[:, 0]
    v011_require(np.all(blocks == breath_ids[:, None]) and len(np.unique(breath_ids)) == 67905, 'Metadata breath grouping invalid')
    rc = np.stack([metadata.R.to_numpy().reshape(-1, 80)[:, 0], metadata.C.to_numpy().reshape(-1, 80)[:, 0]], axis=1)
    for column in ('R', 'C'):
        values = metadata[column].to_numpy().reshape(-1, 80)
        v011_require(np.all(values == values[:, :1]), 'R/C changed within metadata breath')
    with np.load(root/'outputs/improvements_v005/sequence/split_breath_ids.npz', allow_pickle=False) as saved:
        fit = np.sort(pd.Index(breath_ids).get_indexer(saved['A_fit'])).astype(np.int64)
        val = np.sort(pd.Index(breath_ids).get_indexer(saved['A_validation'])).astype(np.int64)
    v011_require((fit >= 0).all() and (val >= 0).all(), 'Missing original A breath IDs')
    split = v011_split_from_known_metadata(breath_ids, rc, fit, val)
    for key in ('A_fit', 'A_validation'):
        v011_require(split['group_hashes'][key] == selection['original_group_hashes'][key], 'Original A group hash differs')
    split_path = out/'D_split.json'
    v011_immutable_json(split_path, split)
    locked = dict(schema='confirmation_v011_locked_before_D', version='v011-r1',
        winner=selection['winner'], baseline=selection['baseline'], required_members=selection['required_members'],
        completed_A_candidates=selection['candidates'], selection_input_files=selection['selection_input_files'],
        notebook_portability_evidence=selection['notebook_portability_evidence'],
        approved_train_sha256=selection['approved_train_sha256'], source_fingerprints=copy.deepcopy(V011_COMPONENTS),
        main_code_sha256=v011_code_sha(root/V011_NOTEBOOK),
        D_split=dict(split_file=v011_relative(root, split_path), split_file_sha256=v011_file_sha(split_path),
            seed=20261202, method=split['method'], counts=split['counts'], group_hashes=split['group_hashes'], index_hashes=split['index_hashes']),
        confirmation_rule=dict(strictly_positive_gain=True, whole_breath_bootstrap_upper95_below_zero=True,
            bootstrap_seed=20261201, bootstrap_draws=1000, baseline_weights=dict(original_compact=.5, deep=.5),
            no_runner_up_after_failure=True, no_weight_or_epoch_search=True),
        original_A_validation_excluded=True, D_is_resampling_confirmation=True,
        pressure_labels_parsed_before_lock=False, D_outcomes_read_before_lock=False)
    v011_immutable_json(out/'locked_before_D.json', locked)
    return locked, split


def v011_verify_lock(root, locked):
    v011_require(v011_code_sha(root/V011_NOTEBOOK) == locked['main_code_sha256'], 'Confirmation source changed after freeze')
    v011_require(v011_file_sha(root/'data/train.csv') == locked['approved_train_sha256'], 'Approved training data changed after freeze')
    v011_source_guard(root, locked['source_fingerprints'])
    for path, digest in locked['selection_input_files'].items():
        v011_require(v011_file_sha(v011_local(root, path)) == digest, 'Frozen A selection input changed: '+path)
    v011_require(v011_file_sha(v011_local(root, locked['D_split']['split_file'])) == locked['D_split']['split_file_sha256'], 'Frozen D split changed')


def v011_member_identity(locked, member):
    return dict(selection_sha256=v011_digest(locked), member_id=member['member_id'],
        factory_id=member['factory_id'], kind=member['kind'], recipe=member['recipe'],
        source_fingerprints=member['source_fingerprints'], main_code_sha256=locked['main_code_sha256'],
        D_fit_group_sha256=locked['D_split']['group_hashes']['D_fit'], training_breaths=54323,
        original_A_validation_excluded=True, cold_initialization=True, validation_feedback=False)


In [ ]:
def v011_metrics(prediction, truth, counts):
    error = np.abs(prediction-truth)
    starts = np.r_[0, np.cumsum(counts)[:-1]]
    return dict(mae=float(error.mean()), median_absolute_error=float(np.median(error)),
        p90_absolute_error=float(np.quantile(error, .9)),
        median_breath_mae=float(np.median(np.add.reduceat(error, starts)/counts)))


def v011_interval(new_error, reference_error, counts):
    sums = np.add.reduceat(new_error-reference_error, np.r_[0, np.cumsum(counts)[:-1]])
    rng = np.random.default_rng(20261201); draws = []
    for _ in range(10):
        take = rng.integers(0, len(counts), size=(100, len(counts)))
        draws.extend((sums[take].sum(axis=1)/counts[take].sum(axis=1)).tolist())
    low, high = np.quantile(draws, [.025, .975])
    return dict(difference=float((new_error-reference_error).mean()),
        bootstrap_95_low=float(low), bootstrap_95_high=float(high), seed=20261201,
        repeats=1000, unit='whole breath', point_count_weighted=True)


def v011_fit_D_member(root, namespace, locked, member, x_fit, y_fit, out_fit, names, resume):
    """Cold fixed-budget fit: deliberately has NO D-validation arguments."""
    v011_verify_lock(root, locked)
    member_id = member['member_id']; directory = root/V011_OUT/'D_models'/member_id
    directory.mkdir(parents=True, exist_ok=True)
    identity = v011_member_identity(locked, member)
    # Also protect legacy V005 resume against a numeric runtime/device change.
    identity['numeric_runtime'] = namespace['_v8_runtime']('cuda')
    v011_immutable_json(directory/'identity.json', identity)
    record_path, report_path, checkpoint = directory/'record.json', directory/'report.json', directory/'model.pt'
    if record_path.exists():
        record = v011_read(record_path)
        v011_require(record['identity'] == identity and record['status'] == 'complete', 'D member identity changed')
        v011_require(v011_file_sha(checkpoint) == record['checkpoint_sha256']
            and v011_file_sha(report_path) == record['report_sha256'], 'Completed D member artifact changed')
        return record
    recipe = member['recipe']; rates = recipe['lr_schedule']; epochs = recipe['selected_epochs']
    v011_require(len(x_fit) == len(y_fit) == len(out_fit) == 54323 and len(rates) == epochs,
        'Wrong frozen D fitting arrays or epoch budget')
    options = dict(epoch_log_path=directory/'epochs.jsonl', feature_names=names,
        device='cuda', resume=resume and namespace['_last_path'](checkpoint).is_file())
    if member['kind'] == 'v5':
        report = namespace['fit_fixed'](x_fit, y_fit, out_fit, recipe['model_spec'], recipe['train_cfg'],
            recipe['seed'], rates, checkpoint, **options)
    else:
        report = namespace['fit_fixed_v008'](x_fit, y_fit, out_fit, recipe['model_spec'], recipe['train_cfg'],
            recipe['seed'], rates, checkpoint, model_factory=v011_factory(member_id, namespace),
            loss_kind=recipe['loss_kind'], arm='v011_D_'+member_id,
            source_fingerprints=member['source_fingerprints'], **options)
    v011_require(report['completed'] and report['executed_epochs'] == epochs
        and report['selected_lr_schedule'] == rates and report['executed_lr_schedule'] == rates
        and report['validation_used'] is False and report['training_breaths'] == 54323,
        'D frozen-budget fit did not complete exactly; resume unchanged')
    saved = namespace['torch'].load(checkpoint, map_location='cpu', weights_only=False)
    v011_require(saved['model_spec'] == recipe['model_spec'] and saved['config'] == recipe['train_cfg']
        and saved['seed'] == recipe['seed'] and saved['fingerprint']['mode'] == 'fixed'
        and saved['fingerprint']['extra'] == dict(lr_schedule=rates)
        and saved['preprocess']['fitted_training_breaths'] == 54323
        and saved['preprocess']['fitted_inspiratory_points'] == int((out_fit == 0).sum())
        and saved['epoch'] == epochs and not saved['validation_used'], 'D fitted statistics or fixed-stage fingerprint differ')
    v011_atomic_json(report_path, report)
    record = dict(status='complete', identity=identity, checkpoint_path=v011_relative(root, checkpoint),
        checkpoint_sha256=v011_file_sha(checkpoint), report_path=v011_relative(root, report_path),
        report_sha256=v011_file_sha(report_path), training_breaths=54323,
        selected_epochs=epochs, validation_feedback=False)
    v011_immutable_json(record_path, record)
    return record


def v011_predict_D_member(root, namespace, member, record, x_val, mask, row_ids):
    """Predictions only; no pressure labels accepted by this function."""
    member_id = member['member_id']; recipe = member['recipe']; checkpoint = v011_local(root, record['checkpoint_path'])
    path = root/V011_OUT/'D_models'/member_id/'D_predictions.npy'
    artifact_path = path.with_name('prediction_artifact.json')
    if artifact_path.exists():
        prior = v011_read(artifact_path)
        prediction = np.load(path, allow_pickle=False)
        v011_require(prediction.dtype == np.float64 and prediction.shape == row_ids.shape
            and np.isfinite(prediction).all() and v011_array_sha(prediction) == prior['prediction_array_sha256']
            and v011_file_sha(path) == prior['prediction_file_sha256']
            and v011_file_sha(checkpoint) == prior['checkpoint_sha256'] == record['checkpoint_sha256']
            and v011_file_sha(v011_local(root, record['report_path'])) == prior['report_sha256'] == record['report_sha256']
            and prior['row_ids_array_sha256'] == v011_array_sha(row_ids)
            and prior['selected_epochs'] == recipe['selected_epochs'], 'Saved D prediction artifact changed')
        return prediction, prior
    v011_require(v011_file_sha(checkpoint) == record['checkpoint_sha256'], 'D model changed before prediction')
    options = dict(device='cuda', batch_size=recipe['train_cfg']['prediction_batch_size'])
    if member['kind'] == 'v5':
        full = namespace['predict_checkpoint'](checkpoint, x_val, **options)
    else:
        full = namespace['predict_checkpoint_v008'](checkpoint, x_val, model_factory=v011_factory(member_id, namespace),
            expected_arm='v011_D_'+member_id, expected_loss_kind=recipe['loss_kind'],
            source_fingerprints=member['source_fingerprints'], **options)
    prediction = full[mask].astype(np.float64)
    v011_require(prediction.shape == row_ids.shape and np.isfinite(prediction).all(), 'D native prediction invalid')
    path = root/V011_OUT/'D_models'/member_id/'D_predictions.npy'
    artifact_path = path.with_name('prediction_artifact.json')
    artifact = dict(member_id=member_id, checkpoint_path=record['checkpoint_path'],
        checkpoint_sha256=record['checkpoint_sha256'], report_path=record['report_path'],
        report_sha256=record['report_sha256'], prediction_file=v011_relative(root, path),
        prediction_array_sha256=v011_array_sha(prediction), row_ids_array_sha256=v011_array_sha(row_ids),
        training_breaths=54323, selected_epochs=recipe['selected_epochs'])
    if artifact_path.exists():
        prior = v011_read(artifact_path)
        v011_require(v011_array_sha(np.load(path, allow_pickle=False)) == prior['prediction_array_sha256']
            == artifact['prediction_array_sha256'], 'Existing D predictions differ')
        artifact['prediction_file_sha256'] = v011_file_sha(path)
        v011_require(prior == artifact, 'D prediction artifact identity differs')
    else:
        v011_save_array(path, prediction)
        artifact['prediction_file_sha256'] = v011_file_sha(path)
        v011_immutable_json(artifact_path, artifact)
    return prediction, artifact


def v011_write_ensemble(root, name, prediction):
    path = root/V011_OUT/(name+'_D_predictions.npy')
    if path.exists():
        v011_require(np.array_equal(np.load(path, allow_pickle=False), prediction), 'Frozen D ensemble differs')
    else:
        v011_save_array(path, prediction)
    return dict(prediction_file=v011_relative(root, path), file_sha256=v011_file_sha(path),
        array_sha256=v011_array_sha(prediction))


def v011_verify_completed_result(root, locked, result):
    v011_require(result['status'] == 'complete' and result['selection_sha256'] == v011_digest(locked)
        and result['locked_before_D_file_sha256'] == v011_file_sha(root/V011_OUT/'locked_before_D.json')
        and result['winner_name'] == locked['winner']['name'], 'Completed D result lock differs')
    v011_require(set(result['member_artifacts']) == {m['member_id'] for m in locked['required_members']}
        and result['all_required_members_completed'] and result['original_A_validation_excluded']
        and result['source_fingerprints'] == locked['source_fingerprints']
        and result['approved_train_sha256'] == locked['approved_train_sha256']
        and result['D_split'] == locked['D_split'], 'Completed D result constraints differ')
    for record in result['member_artifacts'].values():
        for field, digest in [('checkpoint_path', 'checkpoint_sha256'), ('report_path', 'report_sha256'),
                              ('prediction_file', 'prediction_file_sha256')]:
            v011_require(v011_file_sha(v011_local(root, record[field])) == record[digest], 'Completed D member file changed')
        v011_require(v011_array_sha(np.load(v011_local(root, record['prediction_file']), allow_pickle=False))
            == record['prediction_array_sha256'], 'Completed D member prediction changed')
    for record in result['ensemble_artifacts'].values():
        path = record.get('prediction_file', record.get('file'))
        v011_require(v011_file_sha(v011_local(root, path)) == record['file_sha256']
            and v011_array_sha(np.load(v011_local(root, path), allow_pickle=False)) == record['array_sha256'],
            'Completed D ensemble/row artifact changed')
    decision = result['decision']
    v011_require(math.isclose(decision['baseline_MAE']-decision['candidate_MAE'], decision['gain'], abs_tol=1e-12)
        and math.isclose(-decision['gain'], decision['interval']['difference'], abs_tol=1e-12)
        and decision['passed'] == (decision['gain'] > 0. and decision['interval']['bootstrap_95_high'] < 0.),
        'Completed D decision violates frozen gate')


def run_confirmation_v011(root=V011_ROOT, resume=V011_RESUME):
    """Explicit production runner; no final-all fit, submission or alternative recipe."""
    v011_require(os.environ.get('PA3_RUN_CONFIRM_V011', '0') == '1', 'Explicit confirmation RUN approval required')
    root = Path(root).resolve()
    v011_require(root == Path.cwd().resolve(), 'Run confirmation from the repository root')
    own_hash = v011_code_sha(root/V011_NOTEBOOK)
    v011_require(os.environ.get('PA3_CONFIRM_V011_SOURCE_SHA256') == own_hash, 'Exact confirmation source hash required')
    out = root/V011_OUT; out.mkdir(parents=True, exist_ok=True)
    # Shared with the separate final notebook: one approved GPU job at a time.
    gpu_lock = (root/'outputs/.v011_gpu.lock').open('a+')
    try:
        fcntl.flock(gpu_lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        selection = v011_completed_A_selection(root)
        if selection['winner'] is None:
            v011_immutable_json(out/'no_A_qualifier.json', dict(status='no_qualifier',
                candidates=selection['candidates'], selection_input_files=selection['selection_input_files'],
                D_labels_read=False, D_fits_started=False))
            return dict(status='no_qualifier', D_fits_started=False)
        locked, split = v011_freeze_before_D(root, selection)
        v011_verify_lock(root, locked)
        result_path = out/'confirmation_result.json'
        # Existing outcomes are inspected only after recreating and matching the lock.
        if result_path.exists():
            result = v011_read(result_path); v011_verify_completed_result(root, locked, result)
            return result
        namespace = v011_load_components(root)
        v011_require(namespace['torch'].cuda.is_available(), 'Actual approved course CUDA runtime required')
        namespace['seed_sequence'](20261102)
        # FIRST pressure-column parse is after the immutable pre-D lock above.
        frame = pd.read_csv(root/'data/train.csv', dtype=V011_RAW_DTYPES)
        breath_ids = namespace['_validate_feature_frame_v005'](frame)
        y = frame.pressure.to_numpy(copy=True).reshape(-1, 80)
        out_valve = frame.u_out.to_numpy(copy=True).reshape(-1, 80)
        row_ids_all = frame.id.to_numpy(copy=True).reshape(-1, 80)
        known = frame[['id', 'breath_id', 'R', 'C', 'time_step', 'u_in', 'u_out']].copy()
        del frame; gc.collect()
        fit = np.asarray(split['indices']['D_fit'], dtype=np.int64)
        validation = np.asarray(split['indices']['D_validation'], dtype=np.int64)
        original_validation = np.asarray(split['indices']['A_validation'], dtype=np.int64)
        v011_require(v011_array_sha(breath_ids[fit]) == locked['D_split']['group_hashes']['D_fit']
            and v011_array_sha(breath_ids[validation]) == locked['D_split']['group_hashes']['D_validation']
            and not np.intersect1d(np.r_[fit, validation], original_validation).size, 'D groups changed after lock')
        x, names, feature_ids = namespace['make_sequence_features'](known, 'combined')
        v011_require(np.array_equal(feature_ids, breath_ids), 'Feature breath alignment differs')
        del known; gc.collect()
        records = {}
        for member in locked['required_members']:
            # No A checkpoint is passed here: every stage starts from its seeded constructor.
            records[member['member_id']] = v011_fit_D_member(root, namespace, locked, member,
                x[fit], y[fit], out_valve[fit], names, resume)
            gc.collect(); namespace['torch'].cuda.empty_cache()
        mask = out_valve[validation] == 0; counts = mask.sum(axis=1)
        row_ids = row_ids_all[validation][mask]
        v011_require((counts > 0).all() and len(np.unique(row_ids)) == len(row_ids), 'D scoring row alignment invalid')
        row_path = out/'D_row_ids.npy'
        if row_path.exists():
            v011_require(np.array_equal(np.load(row_path, allow_pickle=False), row_ids), 'Saved D row IDs differ')
        else:
            v011_save_array(row_path, row_ids)
        predictions, artifacts = {}, {}
        for member in locked['required_members']:
            member_id = member['member_id']
            predictions[member_id], artifacts[member_id] = v011_predict_D_member(root, namespace,
                member, records[member_id], x[validation], mask, row_ids)
            gc.collect(); namespace['torch'].cuda.empty_cache()
        baseline = sum(member['weight']*predictions[member['member_id']] for member in locked['baseline']['members'])
        candidate = sum(member['weight']*predictions[member['member_id']] for member in locked['winner']['members'])
        # First D metric calculation: after all required fixed-budget fits and predictions.
        truth = y[validation][mask].astype(np.float64)
        baseline_metrics, candidate_metrics = v011_metrics(baseline, truth, counts), v011_metrics(candidate, truth, counts)
        interval = v011_interval(np.abs(candidate-truth), np.abs(baseline-truth), counts)
        gain = baseline_metrics['mae']-candidate_metrics['mae']
        decision = dict(passed=bool(gain > 0. and interval['bootstrap_95_high'] < 0.), gain=float(gain),
            baseline_MAE=baseline_metrics['mae'], candidate_MAE=candidate_metrics['mae'], interval=interval)
        ensemble_artifacts = dict(baseline=v011_write_ensemble(root, 'baseline', baseline),
            candidate=v011_write_ensemble(root, 'candidate', candidate),
            row_ids=dict(file=v011_relative(root, row_path), file_sha256=v011_file_sha(row_path), array_sha256=v011_array_sha(row_ids)))
        result = dict(status='complete', schema='confirmation_v011_result',
            selection_sha256=v011_digest(locked), locked_before_D_file_sha256=v011_file_sha(out/'locked_before_D.json'),
            winner_name=locked['winner']['name'], decision=decision,
            metrics=dict(baseline=baseline_metrics, candidate=candidate_metrics), member_artifacts=artifacts,
            ensemble_artifacts=ensemble_artifacts, approved_train_sha256=locked['approved_train_sha256'],
            source_fingerprints=locked['source_fingerprints'], main_code_sha256=locked['main_code_sha256'],
            D_split=locked['D_split'], D_scored_points=len(truth), D_scored_truth_array_sha256=v011_array_sha(truth),
            all_required_members_completed=True, original_A_validation_excluded=True,
            frozen_A_choice=True, no_runner_up_after_failure=True, confirmation_is_resampling_evidence=True,
            no_D_weight_or_epoch_selection=True, final_all_fitting_performed=False, submission_performed=False,
            official_best_preserved=.18550)
        v011_verify_lock(root, locked)
        v011_immutable_json(result_path, result)
        return result
    finally:
        fcntl.flock(gpu_lock, fcntl.LOCK_UN); gpu_lock.close()


Synthetic protocol checks below use invented groups, errors and temporary files.
They do not access assignment pressure labels, construct the actual D split, load
an A/D model, train, or use a GPU. Run explicitly with `v011_protocol_selfcheck()`.
The production runner remains guarded at the bottom.


In [ ]:
def v011_protocol_selfcheck():
    """Small operational checks only; no real assignment data or models."""
    checks = {}
    def check(name, condition):
        if not bool(condition): raise AssertionError(name)
        checks[name] = True
    def rejects(name, action):
        try: action()
        except (ValueError, FileExistsError): checks[name] = True
        else: raise AssertionError(name)
    with tempfile.TemporaryDirectory(prefix='v011-protocol-') as directory:
        path = Path(directory)/'locked.json'; payload = dict(weights=[.5, .5], seed=20261202)
        v011_immutable_json(path, payload); before = path.read_bytes()
        v011_immutable_json(path, payload)
        check('immutable_exact_rerun_no_change', path.read_bytes() == before)
        rejects('immutable_changed_recipe_rejected', lambda: v011_immutable_json(path, dict(payload, seed=1)))
        check('compact_external_digest', v011_digest(payload) == hashlib.sha256(
            json.dumps(payload, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()).hexdigest())
        check('digest_excludes_self_reference', 'selection_sha256' not in payload)
        # The V010 source manifest, rather than output/metadata bytes, identifies
        # explicitly whitelisted notebook implementations across local/server runs.
        notebook_path = Path(directory)/'controlled_experiments_v008.ipynb'
        original_notebook = dict(cells=[dict(cell_type='code', source=['x = 1\n'],
            execution_count=None, outputs=[])], metadata={})
        notebook_path.write_text(json.dumps(original_notebook))
        expected_raw = v011_file_sha(notebook_path); expected_code = v011_code_sha(notebook_path)
        manifest = {'controlled_experiments_v008.ipynb': dict(file_sha256=expected_raw, code_sha256=expected_code)}
        executed_notebook = copy.deepcopy(original_notebook)
        executed_notebook['metadata'] = dict(executed_by='synthetic proof')
        executed_notebook['cells'][0].update(execution_count=7, outputs=[dict(output_type='stream', text='1\n')])
        notebook_path.write_text(json.dumps(executed_notebook, indent=2))
        evidence = v011_verify_screen_input(Path(directory), notebook_path.name, expected_raw, manifest)
        check('whitelisted_notebook_metadata_only_variation_accepted', evidence['metadata_only_raw_difference']
            and evidence['observed_code_sha256'] == expected_code
            and evidence['expected_raw_file_sha256'] == expected_raw
            and evidence['observed_raw_file_sha256'] == v011_file_sha(notebook_path))
        changed_notebook = copy.deepcopy(executed_notebook)
        changed_notebook['cells'][0]['source'] = ['x = 2\n']
        notebook_path.write_text(json.dumps(changed_notebook))
        rejects('whitelisted_notebook_changed_code_rejected', lambda: v011_verify_screen_input(
            Path(directory), notebook_path.name, expected_raw, manifest))
        artifact = Path(directory)/'synthetic_artifact.bin'; artifact.write_bytes(b'approved')
        expected_artifact = v011_file_sha(artifact); artifact.write_bytes(b'changed')
        rejects('non_notebook_artifact_mismatch_rejected', lambda: v011_verify_screen_input(
            Path(directory), artifact.name, expected_artifact, manifest))
        unlisted = Path(directory)/'unlisted.ipynb'; unlisted.write_text(json.dumps(original_notebook))
        unlisted_expected = v011_file_sha(unlisted); unlisted.write_text(json.dumps(executed_notebook))
        rejects('nonwhitelisted_notebook_metadata_drift_rejected', lambda: v011_verify_screen_input(
            Path(directory), unlisted.name, unlisted_expected,
            {unlisted.name: dict(file_sha256=unlisted_expected, code_sha256=expected_code)}))
    for name, roster in V011_A_ROSTER.items():
        check('fixed_weights_sum_one:'+name, math.isclose(sum(weight for _, weight in roster), 1., abs_tol=1e-15))
    synthetic_candidates = []
    for index, name in enumerate(V011_A_ROSTER):
        mae = .190 if name == 'deep_half_compact_curve_memory_offset' else .192+index*.0001
        gain = .19656260982217663-mae
        synthetic_candidates.append(dict(name=name, A_MAE=mae, A_gain=gain,
            A_interval=dict(bootstrap_95_high=-.001), declared_eligible=bool(gain >= .003)))
    chosen = v011_pick_lowest_A_candidate(synthetic_candidates)
    check('exact_five_candidate_complete_set_required', len(V011_A_ROSTER) == 5)
    check('new_fifth_lowest_A_choice_and_locked_weights', chosen['name'] == 'deep_half_compact_curve_memory_offset'
        and dict(V011_A_ROSTER[chosen['name']]) == dict(deep=.5, original_compact=.125, curve=.125, memory=.125, offset=.125))
    rejects('incomplete_five_candidate_screen_rejected', lambda: v011_pick_lowest_A_candidate(synthetic_candidates[:-1]))
    counts = np.array([1, 3], dtype=np.int64)
    reference_error = np.array([10., 0., 0., 0.]); candidate_error = np.array([0., 1., 1., 1.])
    interval = v011_interval(candidate_error, reference_error, counts)
    check('bootstrap_primary_difference_point_weighted', interval['difference'] == -1.75)
    check('bootstrap_repeat_seed_reproducible', interval == v011_interval(candidate_error, reference_error, counts))
    check('bootstrap_whole_breath_protocol', interval['repeats'] == 1000 and interval['seed'] == 20261201
        and interval['point_count_weighted'] and interval['unit'] == 'whole breath')
    # These invented inputs have the required sizes but are unrelated to assignment IDs.
    fake_ids = np.arange(100000, 167905, dtype=np.int32)
    fake_codes = np.arange(67905) % 9
    fake_rc = np.stack((np.array([5, 20, 50])[fake_codes//3], np.array([10, 20, 50])[fake_codes%3]), axis=1)
    original_fit = np.arange(61114, dtype=np.int64); original_val = np.arange(61114, 67905, dtype=np.int64)
    split = v011_split_from_known_metadata(fake_ids, fake_rc, original_fit, original_val)
    check('synthetic_split_counts_and_exclusion', split['counts']['D_fit'] == 54323 and split['counts']['D_validation'] == 6791
        and not np.intersect1d(split['indices']['D_fit'], original_val).size
        and not np.intersect1d(split['indices']['D_validation'], original_val).size)
    check('synthetic_split_reproducible', split == v011_split_from_known_metadata(fake_ids, fake_rc, original_fit, original_val))
    check('synthetic_indices_sorted', all(values == sorted(values) for values in split['indices'].values()))
    check('split_no_pressure_fields', not any('pressure' in key for key in split))
    return dict(status='passed', checks=checks, check_count=len(checks), synthetic_only=True,
        assignment_data_read=False, actual_D_split_constructed=False, training_performed=False, GPU_work=False)


if V011_RUN:
    V011_RESULT = run_confirmation_v011()
    print(json.dumps(dict(status=V011_RESULT['status'], winner=V011_RESULT.get('winner_name'),
        decision=V011_RESULT.get('decision')), indent=2))
else:
    print('V011 review only: no assignment reads, actual D split, outcomes, fits or final-all work.')
